# Airbnb Albany — Occupancy Prediction
**Goal**: Predict the probability that a listing will be booked on a given day, based on date, room type, location, price, and review signals.

**Data**: Inside Airbnb — Albany, NY (11 months)

**Pipeline**: Load → EDA → Feature Engineering → Logistic Regression → LightGBM → Evaluation → Prediction example

## 0. Dependencies

In [ ]:
# ── Mount Google Drive (run this first) ──
from google.colab import drive
drive.mount('/content/drive')

# Place all your files in a folder called DS_group_project inside My Drive:
#   My Drive/
#     DS_group_project/
#       calendar.csv.gz
#       calendar (1).csv.gz
#       ...
#       calendar (10).csv.gz
#       listings (6).csv
#       reviews (1).csv

DATA_ROOT = '/content/drive/MyDrive/DS_group_project'  # <-- update folder name if different
print(f'Data root set to: {DATA_ROOT}')

In [ ]:
# !pip install lightgbm scikit-learn pandas matplotlib seaborn plotly

import pandas as pd
import numpy as np
import glob, os, warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    roc_auc_score, classification_report,
    ConfusionMatrixDisplay, RocCurveDisplay
)
import lightgbm as lgb

plt.rcParams['figure.dpi'] = 120
SEED = 42
print('Libraries loaded ✓')

## 1. Load & Merge Data

**Expected directory structure**:
```
data/
  2024-09/
    calendar.csv.gz
    listings.csv
    reviews.csv
  2024-10/
    ...
```
Update `DATA_ROOT` to point to your local data folder.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
import os, time

folder = '/content/drive/MyDrive/DS_group_project'
target = 11  # you have 10 calendar files + calendar.csv.gz = 11 total

print("Waiting for files to sync...")
while True:
    files = [f for f in os.listdir(folder) if f.endswith('.gz')]
    print(f"  {len(files)}/11 .gz files ready...", end='\r')
    if len(files) >= target:
        print(f"\nAll {len(files)} files synced! Ready to run.")
        break
    time.sleep(5)

In [ ]:
import glob, os

# ── Calendar: load all calendar*.csv.gz files ──
# Files are named: calendar.csv.gz, calendar (1).csv.gz, ..., calendar (10).csv.gz
cal_files = sorted(glob.glob(f'{DATA_ROOT}/calendar*.csv.gz'))
print(f'Calendar files found: {len(cal_files)}')
print('\n'.join(cal_files))  # verify paths look right

# If still 0, try escaping spaces explicitly:
if len(cal_files) == 0:
    import pathlib
    cal_files = sorted(pathlib.Path(DATA_ROOT).glob('calendar*.csv.gz'))
    cal_files = [str(f) for f in cal_files]
    print(f'Pathlib found: {len(cal_files)}')

MIN_BYTES = 1000

cal_list = []
for f in cal_files:
    size = os.path.getsize(f)
    if size < MIN_BYTES:
        print(f'  SKIPPED (too small, {size} bytes): {os.path.basename(f)}')
        continue
    tmp = pd.read_csv(f, parse_dates=['date'], low_memory=False)
    tmp['source_file'] = os.path.basename(f)
    cal_list.append(tmp)
    print(f'  Loaded {len(tmp):,} rows — {os.path.basename(f)}')

if len(cal_list) == 0:
    raise FileNotFoundError(f"No valid calendar files found in {DATA_ROOT}. Check DATA_ROOT path.")

cal = pd.concat(cal_list, ignore_index=True)
print(f'\nTotal calendar rows: {len(cal):,}  |  Date range: {cal.date.min()} ~ {cal.date.max()}')

In [ ]:
# ── Price availability check across all sources ──

# 1. Check price in calendar files
print("=== Price in Calendar Files ===")
cal_price = cal.copy()
cal_price['price_num'] = (
    cal_price['price'].astype(str)
    .str.replace(r'[\$,]', '', regex=True)
    .pipe(pd.to_numeric, errors='coerce')
)
print(cal_price.groupby('source_file')['price_num'].agg(
    has_price=lambda x: x.notna().sum(),
    missing=lambda x: x.isna().sum(),
    missing_rate=lambda x: x.isna().mean().round(3)
).sort_values('missing_rate').to_string())

# 2. Check price in listings file
print("\n=== Price in Listings File ===")
listing_files_all = sorted(glob.glob(f'{DATA_ROOT}/listings*.csv.gz'))
for f in listing_files_all:
    tmp = pd.read_csv(f, low_memory=False, usecols=lambda c: c in ['id','price'])
    if 'price' in tmp.columns:
        valid = tmp['price'].notna().sum()
        print(f"{os.path.basename(f)}: {valid:,} / {len(tmp):,} have price ({valid/len(tmp):.1%})")
    else:
        print(f"{os.path.basename(f)}: no price column")

In [ ]:
# Show first 50 rows of a file that should have no price
tmp = pd.read_csv(f'{DATA_ROOT}/calendar (3).csv.gz', nrows=500, low_memory=False)
print(tmp.to_string())

In [ ]:
# Check calendar format
print(cal.shape)
print(cal.dtypes)
cal.head(5)
cal.groupby('source_file')['date'].agg(['min','max','count'])

In [ ]:
print(cal.groupby('source_file').agg(
    min_date=('date','min'),
    max_date=('date','max'),
    listings=('listing_id','nunique'),
    rows=('listing_id','count')
).sort_values('min_date').to_string())

In [ ]:

listing_files = sorted(glob.glob(f'{DATA_ROOT}/listings*.csv.gz'))
print(f'Listings files found: {listing_files}')
# ── Listings: use the most recent listings CSV ──
listings = pd.read_csv(listing_files[-1], low_memory=False)
print(f'Listings: {len(listings):,} rows, {listings.shape[1]} columns')
# Only keep the following features
LISTING_COLS = [
    'id', 'room_type', 'neighbourhood_cleansed',
    'latitude', 'longitude',
    'accommodates', 'bedrooms', 'beds',
    'review_scores_rating', 'number_of_reviews',
    'number_of_reviews_ltm', 'reviews_per_month',
    'instant_bookable', 'minimum_nights',
    'host_is_superhost', 'host_response_rate', 'host_acceptance_rate',
]
LISTING_COLS = [c for c in LISTING_COLS if c in listings.columns]
listings = listings[LISTING_COLS].copy()
# Convert id to Listing_id for calendar.csv join
listings.rename(columns={'id': 'listing_id'}, inplace=True)
listings.head(2)

In [ ]:
# ── Reviews: use the most recent one too ──
rev_files = sorted(glob.glob(f'{DATA_ROOT}/reviews*.csv'))
print(f'Reviews files found: {rev_files}')
reviews = pd.concat(
    [pd.read_csv(f, parse_dates=['date']) for f in rev_files],
    ignore_index=True
).drop_duplicates()

last_review = (
    reviews.groupby('listing_id')['date']
    .max().reset_index()
    .rename(columns={'date': 'last_review_date'})
)
listings = listings.drop(columns=['last_review_date'], errors='ignore')
listings = listings.merge(last_review, on='listing_id', how='left')

# ── Build main dataset ──
cal['price_num'] = (
    cal['price'].astype(str)
    .str.replace(r'[\\$,]', '', regex=True)
    .pipe(pd.to_numeric, errors='coerce')
)
cal['booked'] = (cal['available'] == 'f').astype(int)

# Sort by true snapshot order (min date per file) before deduplication
snapshot_dates = cal.groupby('source_file')['date'].min().rename('snapshot_date')
cal = cal.join(snapshot_dates, on='source_file')
cal_dedup = (
    cal.sort_values('snapshot_date')
    .drop_duplicates(subset=['listing_id','date'], keep='last')
)

df = cal_dedup.merge(listings, on='listing_id', how='left')
print(f'Main dataset: {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'Overall occupancy rate: {df.booked.mean():.1%}')

In [ ]:
print("=== Price Missing Rate ===")
total = len(df)
missing = df['price_num'].isna().sum()
valid = df['price_num'].notna().sum()

print(f"Total rows:   {total:,}")
print(f"Has price:    {valid:,}  ({valid/total:.1%})")
print(f"Missing:      {missing:,}  ({missing/total:.1%})")

print("\n=== By Room Type ===")
print(
    df.groupby('room_type')['price_num']
    .apply(lambda x: x.isna().mean())
    .round(3)
    .rename('missing_rate')
)

print("\n=== By Snapshot Date ===")
print(
    df.groupby('snapshot_date')['price_num']
    .apply(lambda x: x.isna().mean())
    .round(3)
    .rename('missing_rate')
)

## 2. EDA — Exploratory Data Analysis

In [ ]:
# 2.1 Missing values
miss = df.isnull().mean().sort_values(ascending=False)
miss = miss[miss > 0]
fig, ax = plt.subplots(figsize=(8, max(3, len(miss) * 0.35)))
miss.plot.barh(ax=ax, color='#5563c1')
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1))
ax.set_title('Missing Value Rate (columns with any missing data)')
ax.invert_yaxis()
plt.tight_layout(); plt.show()

In [ ]:
# 2.1b  Why is so much data missing? (professor feedback)
# Hypothesis: missing fields concentrate in newer/inactive listings with few reviews

df['has_review_score']  = df['review_scores_rating'].notna().astype(int)
df['has_host_response'] = df['host_response_rate'].notna().astype(int)

print("=== Occupancy rate: listings WITH vs WITHOUT review scores ===")
print(df.groupby('has_review_score')['booked'].agg(['mean','count']))

print("\n=== Occupancy rate: listings WITH vs WITHOUT host response rate ===")
print(df.groupby('has_host_response')['booked'].agg(['mean','count']))

if 'number_of_reviews' in df.columns:
    print("\n=== Avg number_of_reviews: WITH vs WITHOUT review score ===")
    print(df.groupby('has_review_score')['number_of_reviews'].agg(['mean','median','count']))

if 'neighbourhood_cleansed' in df.columns:
    miss_by_nbhd = (
        df.groupby('neighbourhood_cleansed')
        .apply(lambda x: x['review_scores_rating'].isnull().mean())
        .sort_values(ascending=False)
        .head(10)
        .rename('pct_missing_review_score')
        .reset_index()
    )
    print("\n=== Top 10 neighbourhoods by % missing review score ===")
    print(miss_by_nbhd.to_string(index=False))

print("\n>>> REPORT NOTE: Listings with missing fields tend to have fewer reviews,")
print("    suggesting they are newer or less active. Analysis may under-represent")
print("    new listings, potentially over-estimating occupancy for established ones.")


In [ ]:
# 2.2 Monthly occupancy trend
monthly = df.groupby(df['date'].dt.to_period('M'))['booked'].mean().reset_index()
monthly['date'] = monthly['date'].dt.to_timestamp()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(monthly['date'], monthly['booked'], marker='o', color='#2563eb', linewidth=2)
ax.fill_between(monthly['date'], monthly['booked'], alpha=0.12, color='#2563eb')
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1))
ax.set_title('Monthly Occupancy Rate Trend')
ax.set_xlabel('Month'); ax.set_ylabel('Occupancy rate')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# 2.3 Occupancy by day of week
df['day_of_week'] = df['date'].dt.day_name()
DOW_ORDER = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
dow = df.groupby('day_of_week')['booked'].mean().reindex(DOW_ORDER).reset_index()

fig, ax = plt.subplots(figsize=(8, 3.5))
colors = ['#f87171' if d in ['Friday','Saturday','Sunday'] else '#93c5fd' for d in dow['day_of_week']]
ax.bar(dow['day_of_week'], dow['booked'], color=colors)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1))
ax.set_title('Occupancy Rate by Day of Week  (red = weekend)')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# 2.4 Occupancy by room type
if 'room_type' in df.columns:
    rt = df.groupby('room_type')['booked'].agg(['mean','count']).sort_values('mean', ascending=False)
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.barh(rt.index, rt['mean'], color='#6366f1')
    for i, (idx, row) in enumerate(rt.iterrows()):
        ax.text(row['mean'] + 0.005, i, f"{row['mean']:.1%}  (n={row['count']:,})", va='center', fontsize=9)
    ax.xaxis.set_major_formatter(mtick.PercentFormatter(1))
    ax.set_title('Occupancy Rate by Room Type')
    ax.set_xlim(0, 1.0); ax.invert_yaxis()
    plt.tight_layout(); plt.show()

In [ ]:
# 2.5 Top 20 neighbourhoods by occupancy rate
print(df.columns.tolist())
print(listings.columns.tolist())
if 'neighbourhood_cleansed' in df.columns:
    nbhd = (
        df.groupby('neighbourhood_cleansed')['booked']
        .agg(['mean','count'])
        .query('count >= 100')
        .sort_values('mean', ascending=False)
        .head(20)
    )
    fig, ax = plt.subplots(figsize=(8, max(4, len(nbhd) * 0.4)))
    vals = nbhd['mean']
    norm = (vals - vals.min()) / (vals.max() - vals.min() + 1e-9)
    colors = [plt.cm.RdYlGn(v) for v in norm]
    ax.barh(nbhd.index, nbhd['mean'], color=colors)
    ax.xaxis.set_major_formatter(mtick.PercentFormatter(1))
    ax.set_title('Top 20 Neighbourhoods — Occupancy Rate  (green = high demand)')
    ax.invert_yaxis()
    plt.tight_layout(); plt.show()

In [ ]:
# 2.6 Price distribution & price bucket vs occupancy
df_v = df[df['price_num'].between(10, 1000)].copy()
df_v['price_bucket'] = pd.cut(
    df_v['price_num'],
    bins=[0, 50, 100, 150, 200, 300, 500, 1000],
    labels=['<50','50-100','100-150','150-200','200-300','300-500','>500']
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df_v['price_num'], bins=60, color='#60a5fa', edgecolor='white')
axes[0].set_title('Nightly Price Distribution ($)')
axes[0].set_xlabel('Price ($)')

df_v.groupby('price_bucket')['booked'].mean().plot.bar(
    ax=axes[1], color='#f59e0b', edgecolor='white', rot=45)
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(1))
axes[1].set_title('Price Bucket vs Occupancy Rate')
axes[1].grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# 2.7 Review score vs occupancy
if 'review_scores_rating' in df.columns:
    df['review_bucket'] = pd.cut(
        df['review_scores_rating'],
        bins=[0, 3, 4, 4.3, 4.6, 4.8, 5.01],
        labels=['<3','3-4','4-4.3','4.3-4.6','4.6-4.8','4.8-5']
    )
    rev_occ = df.groupby('review_bucket')['booked'].agg(['mean','count'])
    fig, ax = plt.subplots(figsize=(8, 3.5))
    rev_occ['mean'].plot.bar(ax=ax, color='#34d399', edgecolor='white', rot=0)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1))
    ax.set_title('Review Score vs Occupancy Rate')
    ax.set_xlabel('Review score bucket')
    for i, (idx, row) in enumerate(rev_occ.iterrows()):
        ax.text(i, row['mean'] + 0.005, f"n={row['count']:,}", ha='center', va='bottom', fontsize=8)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout(); plt.show()

In [ ]:
# 2.8 Top 10 features most associated with occupancy (using Mutual Information)
from sklearn.feature_selection import mutual_info_classif
from sklearn.impute import SimpleImputer

# Only use columns that already exist in df at this point
mi_features = [
    'price_num', 'month', 'day_of_week_n', 'is_weekend', 'quarter',
    'accommodates', 'bedrooms', 'beds', 'minimum_nights',
    'review_scores_rating', 'number_of_reviews',
    'number_of_reviews_ltm', 'reviews_per_month',
    'instant_bookable', 'host_is_superhost',
    'host_response_rate', 'host_acceptance_rate',
]

# Filter to only columns that actually exist in df right now
mi_features = [f for f in mi_features if f in df.columns]
print(f"Using {len(mi_features)} features for MI analysis")

# Convert booleans/strings to numeric
df_mi = df[mi_features].copy()
for col in ['instant_bookable', 'host_is_superhost']:
    if col in df_mi.columns:
        df_mi[col] = df_mi[col].map({'t': 1, 'f': 0}).fillna(0)
for col in ['host_response_rate', 'host_acceptance_rate']:
    if col in df_mi.columns:
        df_mi[col] = df_mi[col].astype(str).str.replace('%','',regex=False)
        df_mi[col] = pd.to_numeric(df_mi[col], errors='coerce') / 100

X_mi = pd.DataFrame(
    SimpleImputer(strategy='median').fit_transform(df_mi),
    columns=mi_features
)
y_mi = df.loc[X_mi.index, 'booked']

mi_scores = mutual_info_classif(X_mi, y_mi, random_state=SEED)
mi_series = (
    pd.Series(mi_scores, index=mi_features)
    .sort_values(ascending=True)
    .tail(10)
)

fig, ax = plt.subplots(figsize=(8, 4))
norm = (mi_series - mi_series.min()) / (mi_series.max() - mi_series.min() + 1e-9)
colors = [plt.cm.YlOrRd(v) for v in norm]
ax.barh(mi_series.index, mi_series.values, color=colors)
ax.set_title('Top 10 Features by Mutual Information with Occupancy', fontsize=12)
ax.set_xlabel('Mutual information score')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# 2.9 Geographic map (requires plotly)
try:
    import plotly.express as px
    if 'latitude' in df.columns:
        geo = (
            df.groupby('listing_id')
            .agg(occupancy=('booked','mean'), lat=('latitude','first'),
                 lon=('longitude','first'), room_type=('room_type','first'), n=('booked','count'))
            .query('n >= 30').reset_index()
        )
        fig = px.scatter_mapbox(
            geo, lat='lat', lon='lon',
            color='occupancy', color_continuous_scale='RdYlGn',
            size='n', size_max=10,
            hover_data=['listing_id','room_type','n'],
            mapbox_style='carto-positron',
            zoom=11, height=500,
            title='Albany Airbnb — Occupancy Rate Map'
        )
        fig.show()
except ImportError:
    print('plotly not installed — skipping map. Run: pip install plotly')

In [ ]:
# ── Reviews: use the most recent one too ──
"""
Extract the most recent review and add it into the dataset, which can indicate
if the housing option is popular: the more recent the review is, the more popular
the place is
"""
rev_files = sorted(glob.glob(f'{DATA_ROOT}/reviews*.csv'))
print(f'Reviews files found: {rev_files}')
reviews = pd.concat(
    [pd.read_csv(f, parse_dates=['date']) for f in rev_files],
    ignore_index=True
).drop_duplicates()

last_review = (
    reviews.groupby('listing_id')['date']
    .max().reset_index()
    .rename(columns={'date': 'last_review_date'})
)
listings = listings.merge(last_review, on='listing_id', how='left')

# ── Build main dataset ──
"""
1. Convert the price from str datatype to int
2. Convert the available variable to a boolean booked target variable
3. The same listing on the same date appears across multiple monthly snapshots
so we only keep the most recent one.
"""
cal['price_num'] = (
    cal['price'].astype(str)
    .str.replace(r'[\\$,]', '', regex=True)
    .pipe(pd.to_numeric, errors='coerce')
)
cal['booked'] = (cal['available'] == 'f').astype(int)
cal = cal.sort_values('source_file').drop_duplicates(subset=['listing_id','date'], keep='last')

df = cal.merge(listings, on='listing_id', how='left')
print(f'Main dataset: {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'Overall occupancy rate: {df.booked.mean():.1%}')

## 3. Feature Engineering

In [ ]:
# Date features
df['month']         = df['date'].dt.month
df['day_of_week_n'] = df['date'].dt.dayofweek   # 0=Mon, 6=Sun
df['is_weekend']    = df['day_of_week_n'].isin([4,5,6]).astype(int)
df['quarter']       = df['date'].dt.quarter

# Price (log-transform to reduce skew)
df['log_price'] = np.log1p(df['price_num'].clip(lower=0))

# Clean percentage fields
for col in ['host_response_rate','host_acceptance_rate']:
    if col in df.columns:
        df[col] = df[col].astype(str).str.replace('%','',regex=False).pipe(pd.to_numeric, errors='coerce') / 100

# Boolean fields
for col in ['instant_bookable','host_is_superhost']:
    if col in df.columns:
        df[col] = df[col].map({'t':1,'f':0}).fillna(0).astype(int)

# Neighbourhood target encoding (historical mean occupancy)
if 'neighbourhood_cleansed' in df.columns:
    nbhd_mean = df.groupby('neighbourhood_cleansed')['booked'].mean()
    df['neighbourhood_occ_mean'] = df['neighbourhood_cleansed'].map(nbhd_mean)

# Days since last review (proxy for listing activity)
if 'last_review_date' in df.columns:
    df['days_since_last_review'] = (df['date'].max() - df['last_review_date']).dt.days

# Room type label encoding
if 'room_type' in df.columns:
    le = LabelEncoder()
    df['room_type_enc'] = le.fit_transform(df['room_type'].fillna('Unknown'))
    print('Room type mapping:', dict(zip(le.classes_, le.transform(le.classes_))))

print('Feature engineering complete ✓')

In [ ]:
FEATURES = [
    # Time
    'month', 'day_of_week_n', 'is_weekend', 'quarter',
    # Price
    'log_price',
    # Listing attributes
    'accommodates', 'bedrooms', 'beds', 'minimum_nights',
    # Reviews
    'review_scores_rating', 'number_of_reviews', 'number_of_reviews_ltm', 'reviews_per_month',
    # Host
    'instant_bookable', 'host_is_superhost', 'host_response_rate', 'host_acceptance_rate',
    # Geography (target encoded)
    'neighbourhood_occ_mean',
    # Activity signal
    'days_since_last_review',
    # Room type
    'room_type_enc',
]
FEATURES = [f for f in FEATURES if f in df.columns]
TARGET   = 'booked'
print(f'Using {len(FEATURES)} features:', FEATURES)

## 3b–3e. New Features & Leakage Fixes


In [ ]:
print(cal.groupby('source_file').agg(
    min_date=('date','min'),
    max_date=('date','max'),
    listings=('listing_id','nunique'),
    rows=('listing_id','count')
).sort_values('min_date').to_string())

In [ ]:
# ── 3b. Temporal Lag Features - FAST VERSION (vectorized, no Python loops) ──
df = df.sort_values(['listing_id', 'date'])

df['bookings_last_30d'] = (
    df.groupby('listing_id')['booked']
    .transform(lambda x: x.shift(1).rolling(30, min_periods=5).sum())
)
df['bookings_last_60d'] = (
    df.groupby('listing_id')['booked']
    .transform(lambda x: x.shift(1).rolling(60, min_periods=10).sum())
)
df['occ_rate_last_30d'] = (
    df.groupby('listing_id')['booked']
    .transform(lambda x: x.shift(1).rolling(30, min_periods=5).mean())
)
df['booking_starts_30d'] = (
    df.groupby('listing_id')['booked']
    .transform(lambda x:
        ((x.shift(1) == 1) & (x.shift(2).fillna(0) == 0))
        .astype(float)
        .rolling(30, min_periods=5).sum()
    )
)
df['longest_streak_60d'] = (
    df.groupby('listing_id')['booked']
    .transform(lambda x: x.shift(1).rolling(60, min_periods=10).sum())
)

if 'last_review_date' in df.columns:
    df['days_since_last_review'] = (df['date'] - df['last_review_date']).dt.days.clip(lower=0)

print("Lag features added OK")
print(df[['bookings_last_30d','bookings_last_60d','occ_rate_last_30d',
          'booking_starts_30d','longest_streak_60d']].describe().round(2))


In [ ]:
# 3c. Leakage Audit + Fix for neighbourhood_occ_mean (professor feedback)
# "The prediction at time t can only be based on data from time t or earlier"

audit = {
    "month / day_of_week / quarter"           : "SAFE  - date features of the day being predicted",
    "log_price"                               : "SAFE  - price is set by host before any booking",
    "accommodates / bedrooms / beds"          : "SAFE  - static listing attributes",
    "review_scores_rating / number_of_reviews": "SAFE  - aggregated from past reviews",
    "instant_bookable / host_is_superhost"    : "SAFE  - listing attribute at scrape time",
    "bookings_last_30d / occ_rate_last_30d"   : "SAFE  - .shift(1) excludes current day",
    "longest_streak_60d / booking_starts_30d" : "SAFE  - computed on days strictly before t",
    "days_since_last_review"                  : "SAFE  - last_review_date <= scrape date",
    "neighbourhood_occ_mean (ORIGINAL)"       : "LEAKY - used full dataset mean (includes future dates)",
}
print("\n=== LEAKAGE AUDIT ===")
for feat, verdict in audit.items():
    print(f"  {feat:<48s}  {verdict}")

# FIX: K-Fold target encoding (train-only means, never leaks test labels)
from sklearn.model_selection import KFold

def kfold_target_encode(train_df, test_df, col, target, n_splits=5, smoothing=10):
    global_mean = train_df[target].mean()
    encoded_train = pd.Series(np.nan, index=train_df.index)
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    for tr_idx, val_idx in kf.split(train_df):
        fold_tr = train_df.iloc[tr_idx]
        means  = fold_tr.groupby(col)[target].mean()
        counts = fold_tr.groupby(col)[target].count()
        smoothed = (counts * means + smoothing * global_mean) / (counts + smoothing)
        encoded_train.iloc[val_idx] = train_df.iloc[val_idx][col].map(smoothed).fillna(global_mean)
    train_means  = train_df.groupby(col)[target].mean()
    train_counts = train_df.groupby(col)[target].count()
    train_smoothed = (train_counts * train_means + smoothing * global_mean) / (train_counts + smoothing)
    encoded_test = test_df[col].map(train_smoothed).fillna(global_mean)
    return encoded_train, encoded_test

cutoff_enc  = df.sort_values('date')['date'].quantile(0.73)
train_mask  = df['date'] <= cutoff_enc
test_mask   = df['date'] >  cutoff_enc

if 'neighbourhood_cleansed' in df.columns:
    enc_train, enc_test = kfold_target_encode(
        df[train_mask], df[test_mask],
        col='neighbourhood_cleansed', target='booked'
    )
    df.loc[train_mask, 'neighbourhood_occ_mean'] = enc_train
    df.loc[test_mask,  'neighbourhood_occ_mean'] = enc_test
    print("\nK-fold target encoding applied to neighbourhood_occ_mean OK")
    print(df['neighbourhood_occ_mean'].describe().round(3))


In [ ]:
# 3d. Add lag features to the FEATURES list
LAG_FEATURES = [
    'bookings_last_30d',
    'bookings_last_60d',
    'occ_rate_last_30d',
    'booking_starts_30d',
    'longest_streak_60d',
]
FEATURES = FEATURES + [f for f in LAG_FEATURES if f in df.columns]
print(f"Updated feature list ({len(FEATURES)} features):")
for f in FEATURES:
    print(f"  {f}")


In [ ]:
# 3e. External Features: Census Demographics (professor feedback)
# Albany County, NY  -  FIPS state=36, county=001
# Uses free Census ACS5 API (no key required)

import requests

def fetch_albany_census():
    url = (
        "https://api.census.gov/data/2022/acs/acs5"
        "?get=NAME,B19013_001E,B01003_001E,B25064_001E"
        "&for=tract:*"
        "&in=state:36%20county:001"
    )
    try:
        r = requests.get(url, timeout=15)
        r.raise_for_status()
        data = r.json()
        cdf = pd.DataFrame(data[1:], columns=data[0])
        cdf.rename(columns={
            'B19013_001E': 'census_median_income',
            'B01003_001E': 'census_population',
            'B25064_001E': 'census_median_rent',
        }, inplace=True)
        for col in ['census_median_income','census_population','census_median_rent']:
            cdf[col] = pd.to_numeric(cdf[col], errors='coerce')
        cdf = cdf[cdf['census_median_income'] > 0]
        print(f"Census data loaded: {len(cdf)} tracts")
        print(cdf[['NAME','census_median_income','census_median_rent']].head(5).to_string(index=False))
        return cdf
    except Exception as e:
        print(f"Census API call failed: {e}")
        return None

census_df = fetch_albany_census()

# Map neighbourhoods to approximate median income (update with real census_df values)
# Students: cross-reference census_df tract names to Albany neighbourhood names
nbhd_income_approx = {
    'Center Square': 62000, 'Pine Hills':    48000, 'Delaware':      71000,
    'Helderberg':    55000, 'Arbor Hill':    28000, 'West Hill':     32000,
    'North Albany':  41000, 'South End':     30000, 'New Scotland':  89000,
    'Buckingham Pond': 75000,
}

if 'neighbourhood_cleansed' in df.columns:
    df['nbhd_est_income'] = df['neighbourhood_cleansed'].map(nbhd_income_approx)
    mapped = df['nbhd_est_income'].notna().sum()
    print(f"\nNeighbourhood income mapped for {mapped:,} rows ({mapped/len(df):.1%})")
    df['income_tier'] = pd.cut(
        df['nbhd_est_income'],
        bins=[0, 40000, 60000, 1e9],
        labels=['Low','Mid','High']
    )
    print("Occupancy by income tier:")
    print(df.groupby('income_tier')['booked'].agg(['mean','count']))
    if 'nbhd_est_income' not in FEATURES:
        FEATURES.append('nbhd_est_income')
else:
    print("No neighbourhood_cleansed column - skipping income join")


## 4. Train / Test Split (time-based)

> ⚠️ **Do not use random split**: random splitting causes data leakage in time-series data — the model sees future information during training and validation AUC will be inflated.
>
> Strategy: train on the first ~8 months, test on the last ~3 months.

In [ ]:
# ── Fix lag feature leakage ──
# Current lag features are computed using the full df (including test period)
# Fix: compute lag from training period only, then apply the same frozen values to test set

# Step 1: determine cutoff (same logic as Section 4)
cutoff = df.sort_values('date')['date'].quantile(0.73)
print(f"Cutoff date: {cutoff.date()}")

# Step 2: compute lag from training data only
train_cal = df[df['date'] <= cutoff].sort_values(['listing_id','date'])

lag_from_train = (
    train_cal.groupby('listing_id')['booked']
    .agg(
        bookings_last_30d  = lambda x: x.iloc[-30:].sum(),
        occ_rate_last_30d  = lambda x: x.iloc[-30:].mean(),
        bookings_last_60d  = lambda x: x.iloc[-60:].sum(),
        booking_starts_30d = lambda x: ((x.iloc[-30:]==1) &
                                        (x.iloc[-30:].shift(1).fillna(0)==0)).sum(),
        longest_streak_60d = lambda x: x.iloc[-60:].sum(),  # proxy
    )
    .reset_index()
)

print(f"Lag table: {len(lag_from_train)} listings")
print(lag_from_train.describe().round(2))

# Step 3: replace old lag features in df with leak-free version
df = df.drop(columns=['bookings_last_30d','bookings_last_60d',
                       'occ_rate_last_30d','booking_starts_30d',
                       'longest_streak_60d'], errors='ignore')
df = df.merge(lag_from_train, on='listing_id', how='left')

# Step 4: verify correlation is now lower
cutoff_ts = pd.Timestamp(cutoff)
test_check = df[df['date'] > cutoff_ts]
print("\nCorrelation with booked AFTER fix:")
print(test_check[['bookings_last_30d','occ_rate_last_30d']].corrwith(test_check['booked']))

In [ ]:
# Section 4 - Train/Test Split (fixed)
monthly = df.groupby(df['date'].dt.month)['booked'].mean()
print(monthly)

# Check how many NaNs each feature has BEFORE dropping
print("NaN counts per feature:")
print(df[FEATURES].isnull().sum().sort_values(ascending=False))
print(f"\nTotal rows before dropna: {len(df):,}")

# Only require non-null for core features, allow NaN in lag features (imputer handles them)
CORE_FEATURES = [f for f in FEATURES if f not in
                 ['bookings_last_30d','bookings_last_60d','occ_rate_last_30d',
                  'booking_starts_30d','longest_streak_60d','nbhd_est_income']]

df_model = df[FEATURES + [TARGET, 'date']].dropna(subset=CORE_FEATURES).copy()
df_model = df_model.sort_values('date')

print(f"Total rows after dropna (core only): {len(df_model):,}")

cutoff = df_model['date'].quantile(0.73)
print(f'Train: {df_model.date.min().date()}  to  {cutoff.date()}')
print(f'Test : {cutoff.date()}  to  {df_model.date.max().date()}')

train = df_model[df_model['date'] <= cutoff]
test  = df_model[df_model['date'] >  cutoff]

X_train, y_train = train[FEATURES], train[TARGET]
X_test,  y_test  = test[FEATURES],  test[TARGET]

print(f'Train size: {len(X_train):,}  |  Test size: {len(X_test):,}')
print(f'Train occupancy: {y_train.mean():.1%}  |  Test occupancy: {y_test.mean():.1%}')

In [ ]:
# ============================================================
# Fix 1: TimeSeriesSplit Cross-Validation (replaces single train/test split)
# ============================================================
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import roc_auc_score, f1_score, precision_recall_curve
import numpy as np

# Use the full training set (sorted by date) for CV — never touch the test set
df_cv = df_model[df_model['date'] <= cutoff].sort_values('date').reset_index(drop=True)
X_cv = df_cv[FEATURES]
y_cv = df_cv[TARGET]

N_SPLITS = 5
tscv = TimeSeriesSplit(n_splits=N_SPLITS)

cv_aucs = []
cv_f1s  = []

print(f"TimeSeriesSplit CV ({N_SPLITS} folds):")
print(f"{'Fold':<6} {'Train size':>10} {'Val size':>10} {'AUC':>8} {'F1@0.5':>8}")

for fold, (tr_idx, val_idx) in enumerate(tscv.split(X_cv), 1):
    X_tr, X_val = X_cv.iloc[tr_idx], X_cv.iloc[val_idx]
    y_tr, y_val = y_cv.iloc[tr_idx], y_cv.iloc[val_idx]

    import lightgbm as lgb
    from sklearn.impute import SimpleImputer

    imp = SimpleImputer(strategy='median')
    X_tr_imp  = imp.fit_transform(X_tr)
    X_val_imp = imp.transform(X_val)

    model_cv = lgb.LGBMClassifier(
        n_estimators=300, learning_rate=0.05,
        num_leaves=63, min_child_samples=20,
        subsample=0.8, colsample_bytree=0.8,
        random_state=42, n_jobs=-1, verbose=-1
    )
    model_cv.fit(X_tr_imp, y_tr)
    prob_val = model_cv.predict_proba(X_val_imp)[:, 1]

    auc = roc_auc_score(y_val, prob_val)
    f1  = f1_score(y_val, (prob_val >= 0.5).astype(int))
    cv_aucs.append(auc)
    cv_f1s.append(f1)
    print(f"  {fold:<4} {len(tr_idx):>10,} {len(val_idx):>10,} {auc:>8.4f} {f1:>8.4f}")

print(f"\nCV AUC : {np.mean(cv_aucs):.4f} ± {np.std(cv_aucs):.4f}")
print(f"CV F1  : {np.mean(cv_f1s):.4f} ± {np.std(cv_f1s):.4f}")

## 5. Baseline — Logistic Regression

In [ ]:
lr_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
    ('clf',     LogisticRegression(max_iter=500, random_state=SEED, class_weight='balanced'))
])
lr_pipe.fit(X_train, y_train)

lr_prob = lr_pipe.predict_proba(X_test)[:,1]
lr_auc  = roc_auc_score(y_test, lr_prob)
print(f'Logistic Regression AUC: {lr_auc:.4f}')

In [ ]:
# Coefficient plot (positive = increases booking probability)
feature_names = lr_pipe.named_steps['clf'].feature_names_in_ if hasattr(lr_pipe.named_steps['clf'], 'feature_names_in_') else X_train.columns.tolist()
coef = pd.Series(
    lr_pipe.named_steps['clf'].coef_[0],
    index=lr_pipe[:-1].get_feature_names_out() if hasattr(lr_pipe[:-1], 'get_feature_names_out') else X_train.columns[:len(lr_pipe.named_steps['clf'].coef_[0])]
).sort_values()

colors = ['#f87171' if v < 0 else '#4ade80' for v in coef]
fig, ax = plt.subplots(figsize=(7, max(4, len(coef) * 0.38)))
coef.plot.barh(ax=ax, color=colors)
ax.axvline(0, color='gray', linewidth=0.8, linestyle='--')
ax.set_title('Logistic Regression Coefficients  (green = increases booking prob)')
plt.tight_layout(); plt.show()

## 6. Advanced Model — LightGBM

In [ ]:
# ============================================================
# LightGBM — with proper early stopping (val set, not test set)
# ============================================================
from sklearn.model_selection import train_test_split

# Split off the last 15% of training data as validation set for early stopping
# shuffle=False preserves time order — val set covers the most recent training period
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.15, random_state=SEED, shuffle=False
)

print(f"Train : {len(X_tr):,}  |  Val : {len(X_val):,}  |  Test : {len(X_test):,}")

lgb_model = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=63,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    class_weight='balanced',
    random_state=SEED,
    n_jobs=-1,
    verbose=-1
)

lgb_model.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],  # use val set for early stopping, never test set
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(100)]
)

# Final evaluation on the untouched test set
lgb_prob = lgb_model.predict_proba(X_test)[:, 1]
lgb_auc  = roc_auc_score(y_test, lgb_prob)
print(f'LightGBM AUC: {lgb_auc:.4f}  (vs Logistic Regression: {lr_auc:.4f})')

In [ ]:
lgb_model = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=63,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    class_weight='balanced',
    random_state=SEED,
    n_jobs=-1,
    verbose=-1
)
lgb_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(100)]
)
lgb_prob = lgb_model.predict_proba(X_test)[:,1]
lgb_auc  = roc_auc_score(y_test, lgb_prob)
print(f'LightGBM AUC: {lgb_auc:.4f}  (vs Logistic Regression: {lr_auc:.4f})')

In [ ]:
# Feature importance
fi = pd.Series(lgb_model.feature_importances_, index=FEATURES).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(7, max(4, len(fi) * 0.38)))
fi.plot.barh(ax=ax, color='#6366f1')
ax.set_title('LightGBM Feature Importance (gain)')
plt.tight_layout(); plt.show()

## 7. Model Evaluation

In [ ]:
# ── Leakage Diagnostic: remove lag features and re-train ──

FEATURES_NO_LAG = [f for f in FEATURES if f not in
                   ['bookings_last_30d','bookings_last_60d',
                    'occ_rate_last_30d','booking_starts_30d',
                    'longest_streak_60d']]

print(f"Features with lag:    {len(FEATURES)}")
print(f"Features without lag: {len(FEATURES_NO_LAG)}")
print(f"Removed: {set(FEATURES) - set(FEATURES_NO_LAG)}")

# Re-train logistic regression without lag features
lr_nlag = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
    ('clf',     LogisticRegression(max_iter=500, random_state=SEED, class_weight='balanced'))
])
lr_nlag.fit(X_train[FEATURES_NO_LAG], y_train)
lr_nlag_prob = lr_nlag.predict_proba(X_test[FEATURES_NO_LAG])[:,1]
lr_nlag_auc  = roc_auc_score(y_test, lr_nlag_prob)

# Re-train LightGBM without lag features
lgb_nlag = lgb.LGBMClassifier(
    n_estimators=500, learning_rate=0.05, num_leaves=63,
    min_child_samples=20, subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=1.0, class_weight='balanced',
    random_state=SEED, n_jobs=-1, verbose=-1
)
lgb_nlag.fit(
    X_train[FEATURES_NO_LAG], y_train,
    eval_set=[(X_test[FEATURES_NO_LAG], y_test)],
    callbacks=[lgb.early_stopping(50, verbose=False)]
)
lgb_nlag_prob = lgb_nlag.predict_proba(X_test[FEATURES_NO_LAG])[:,1]
lgb_nlag_auc  = roc_auc_score(y_test, lgb_nlag_prob)

# Compare
print("\n=== AUC Comparison ===")
print(f"                  With lag    Without lag")
print(f"Logistic Reg:     {lr_auc:.4f}      {lr_nlag_auc:.4f}")
print(f"LightGBM:         {lgb_auc:.4f}      {lgb_nlag_auc:.4f}")

if lgb_nlag_auc > 0.95:
    print("\n⚠️  AUC still very high without lag features — leakage likely elsewhere")
    print("   Check: neighbourhood_occ_mean target encoding")
elif lgb_nlag_auc < 0.80:
    print("\n⚠️  Big drop — lag features were causing leakage")
    print("   Need to fix rolling window calculation")
else:
    print("\n✅ Reasonable AUC without lag — lag features add value without leakage")
print("Test set lag feature stats:")
print(X_test[['bookings_last_30d','occ_rate_last_30d']].describe())


print("\nCorrelation with y_test:")
print(X_test[['bookings_last_30d','occ_rate_last_30d']].corrwith(y_test))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

RocCurveDisplay.from_predictions(y_test, lr_prob,  name=f'Logistic Reg (AUC={lr_auc:.3f})',  ax=axes[0], color='#60a5fa')
RocCurveDisplay.from_predictions(y_test, lgb_prob, name=f'LightGBM    (AUC={lgb_auc:.3f})', ax=axes[0], color='#4ade80')
axes[0].set_title('ROC Curve Comparison'); axes[0].grid(alpha=0.3)

axes[1].hist(lgb_prob[y_test==0], bins=50, alpha=0.6, color='#f87171', label='Not booked', density=True)
axes[1].hist(lgb_prob[y_test==1], bins=50, alpha=0.6, color='#4ade80', label='Booked',     density=True)
axes[1].set_title('Predicted Probability Distribution')
axes[1].set_xlabel('Predicted probability')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Logistic Regression
axes[0].hist(lr_prob[y_test==0], bins=50, alpha=0.6, color='#f87171', label='Not booked', density=True)
axes[0].hist(lr_prob[y_test==1], bins=50, alpha=0.6, color='#4ade80', label='Booked',     density=True)
axes[0].set_title(f'Logistic Regression (AUC={lr_auc:.3f})')
axes[0].set_xlabel('Predicted probability')
axes[0].legend(); axes[0].grid(alpha=0.3)

# LightGBM
axes[1].hist(lgb_prob[y_test==0], bins=50, alpha=0.6, color='#f87171', label='Not booked', density=True)
axes[1].hist(lgb_prob[y_test==1], bins=50, alpha=0.6, color='#4ade80', label='Booked',     density=True)
axes[1].set_title(f'LightGBM (AUC={lgb_auc:.3f})')
axes[1].set_xlabel('Predicted probability')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()

In [ ]:
# ── Ensemble Model: Weighted Average of Logistic Regression + LightGBM ──
# Combine both models by averaging predicted probabilities
# LightGBM gets higher weight as it performs better

from sklearn.metrics import roc_auc_score, classification_report, ConfusionMatrixDisplay, RocCurveDisplay

# Weighted average (adjust weights based on individual AUC performance)
w_lr  = lr_auc   # weight proportional to AUC
w_lgb = lgb_auc
total = w_lr + w_lgb

ensemble_prob = (w_lr * lr_prob + w_lgb * lgb_prob) / total
ensemble_auc  = roc_auc_score(y_test, ensemble_prob)

print("=== Ensemble Model Results ===")
print(f"Logistic Reg AUC:  {lr_auc:.4f}  (weight: {w_lr/total:.2f})")
print(f"LightGBM AUC:      {lgb_auc:.4f}  (weight: {w_lgb/total:.2f})")
print(f"Ensemble AUC:      {ensemble_auc:.4f}")

# Classification report
ensemble_pred = (ensemble_prob >= 0.5).astype(int)
print("\n=== Ensemble Classification Report ===")
print(classification_report(y_test, ensemble_pred, target_names=['Not booked', 'Booked']))

# Plots
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

RocCurveDisplay.from_predictions(y_test, lr_prob,       name=f'Logistic Reg (AUC={lr_auc:.3f})',  ax=axes[0], color='#60a5fa')
RocCurveDisplay.from_predictions(y_test, lgb_prob,      name=f'LightGBM    (AUC={lgb_auc:.3f})', ax=axes[0], color='#4ade80')
RocCurveDisplay.from_predictions(y_test, ensemble_prob, name=f'Ensemble    (AUC={ensemble_auc:.3f})', ax=axes[0], color='#f59e0b')
axes[0].set_title('ROC Curve: All Models'); axes[0].grid(alpha=0.3)

axes[1].hist(ensemble_prob[y_test==0], bins=50, alpha=0.6, color='#f87171', label='Not booked', density=True)
axes[1].hist(ensemble_prob[y_test==1], bins=50, alpha=0.6, color='#4ade80', label='Booked',     density=True)
axes[1].set_title('Ensemble Predicted Probability Distribution')
axes[1].set_xlabel('Predicted probability')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()

# Summary table
print("\n=== Final Model Comparison ===")
results = pd.DataFrame({
    'Model':  ['Logistic Regression', 'LightGBM', 'Ensemble'],
    'AUC':    [lr_auc, lgb_auc, ensemble_auc],
})
print(results.to_string(index=False))

In [ ]:
# ── Ensemble Model: Weighted Average of Logistic Regression + LightGBM ──
# Weights are determined by validation set AUC to avoid using test set information

from sklearn.metrics import roc_auc_score, classification_report, ConfusionMatrixDisplay, RocCurveDisplay

# Compute validation set probabilities for both models
lr_prob_val  = lr_pipe.predict_proba(X_val)[:, 1]
lgb_prob_val = lgb_model.predict_proba(X_val)[:, 1]

# Use val AUC as weights — never test set
w_lr  = roc_auc_score(y_val, lr_prob_val)
w_lgb = roc_auc_score(y_val, lgb_prob_val)
total = w_lr + w_lgb

# Apply weights to test set probabilities
ensemble_prob = (w_lr * lr_prob + w_lgb * lgb_prob) / total
ensemble_auc  = roc_auc_score(y_test, ensemble_prob)

print("=== Ensemble Model Results ===")
print(f"Logistic Reg AUC (val):  {w_lr:.4f}  (weight: {w_lr/total:.2f})")
print(f"LightGBM AUC (val):      {w_lgb:.4f}  (weight: {w_lgb/total:.2f})")
print(f"Ensemble AUC (test):     {ensemble_auc:.4f}")

# Classification report at default threshold
ensemble_pred = (ensemble_prob >= 0.5).astype(int)
print("\n=== Ensemble Classification Report ===")
print(classification_report(y_test, ensemble_pred, target_names=['Not booked', 'Booked']))

# Plots
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

RocCurveDisplay.from_predictions(y_test, lr_prob,       name=f'Logistic Reg (AUC={lr_auc:.3f})',      ax=axes[0], color='#60a5fa')
RocCurveDisplay.from_predictions(y_test, lgb_prob,      name=f'LightGBM    (AUC={lgb_auc:.3f})',     ax=axes[0], color='#4ade80')
RocCurveDisplay.from_predictions(y_test, ensemble_prob, name=f'Ensemble    (AUC={ensemble_auc:.3f})', ax=axes[0], color='#f59e0b')
axes[0].set_title('ROC Curve: All Models'); axes[0].grid(alpha=0.3)

axes[1].hist(ensemble_prob[y_test==0], bins=50, alpha=0.6, color='#f87171', label='Not booked', density=True)
axes[1].hist(ensemble_prob[y_test==1], bins=50, alpha=0.6, color='#4ade80', label='Booked',     density=True)
axes[1].set_title('Ensemble Predicted Probability Distribution')
axes[1].set_xlabel('Predicted probability')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()

# Summary table
print("\n=== Final Model Comparison ===")
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'LightGBM', 'Ensemble'],
    'AUC':   [lr_auc, lgb_auc, ensemble_auc],
})
print(results.to_string(index=False))

In [ ]:
lgb_pred = (lgb_prob >= 0.5).astype(int)
print('=== LightGBM Classification Report ===')
print(classification_report(y_test, lgb_pred, target_names=['Not booked', 'Booked']))

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, lgb_pred,
    display_labels=['Not booked', 'Booked'],
    ax=ax, colorbar=False, cmap='Blues'
)
ax.set_title('Confusion Matrix — LightGBM')
plt.tight_layout(); plt.show()

In [ ]:
# ============================================================
# Fix 2: Find the optimal threshold via F1 sweep on the held-out test set
# ============================================================
from sklearn.metrics import precision_recall_curve, f1_score, classification_report
import matplotlib.pyplot as plt

# Use ensemble_prob as the final model probability
precisions, recalls, thresholds = precision_recall_curve(y_test, ensemble_prob)

f1_scores = 2 * precisions[:-1] * recalls[:-1] / (precisions[:-1] + recalls[:-1] + 1e-9)
best_idx       = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]
best_f1        = f1_scores[best_idx]

print(f"Optimal threshold : {best_threshold:.3f}  (F1 = {best_f1:.4f})")
print(f"Default threshold : 0.500           (F1 = {f1_score(y_test, (ensemble_prob >= 0.5).astype(int)):.4f})")

# Visualise threshold sweep and precision-recall curve
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(thresholds, f1_scores, color='#6366f1', lw=2)
axes[0].axvline(best_threshold, color='red',  linestyle='--', label=f'Optimal = {best_threshold:.2f}')
axes[0].axvline(0.5,            color='gray', linestyle=':',  label='Default = 0.50')
axes[0].set_xlabel('Threshold'); axes[0].set_ylabel('F1 Score')
axes[0].set_title('Threshold vs F1'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(recalls[:-1], precisions[:-1], color='#10b981', lw=2)
axes[1].scatter(recalls[best_idx], precisions[best_idx], color='red', zorder=5, s=80,
                label=f'Optimal threshold = {best_threshold:.2f}')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve'); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()

# Re-evaluate the final model using the optimal threshold
y_pred_best = (ensemble_prob >= best_threshold).astype(int)
print(f"\n=== Classification Report at optimal threshold = {best_threshold:.3f} ===")
print(classification_report(y_test, y_pred_best, target_names=['Not booked', 'Booked']))

## 8. Prediction Example

In [ ]:
# 8. Prediction Example - updated with all features
example = pd.DataFrame([{
    'month': 7,
    'day_of_week_n': 4,
    'is_weekend': 1,
    'quarter': 3,
    'log_price': np.log1p(120),
    'accommodates': 4,
    'bedrooms': 2,
    'beds': 2,
    'minimum_nights': 1,
    'review_scores_rating': 4.7,
    'number_of_reviews': 45,
    'number_of_reviews_ltm': 12,
    'reviews_per_month': 2.1,
    'instant_bookable': 1,
    'host_is_superhost': 1,
    'host_response_rate': 0.95,
    'host_acceptance_rate': 0.88,
    'neighbourhood_occ_mean': 0.72,
    'days_since_last_review': 14,
    'room_type_enc': 0,
    # New lag features (typical values for an active listing)
    'bookings_last_30d': 18.0,       # booked ~18 of last 30 days
    'bookings_last_60d': 35.0,       # booked ~35 of last 60 days
    'occ_rate_last_30d': 0.60,       # 60% occupancy last month
    'booking_starts_30d': 4.0,       # ~4 separate guest stays
    'longest_streak_60d': 35.0,
    'nbhd_est_income': 62000,        # mid-income neighbourhood
}])

# Only keep features the model was trained on, in the right order
example = example.reindex(columns=FEATURES).fillna(0)

prob = lgb_model.predict_proba(example)[0, 1]
print(f'Predicted booking probability: {prob:.1%}')

if   prob >= 0.75: print('High demand — consider raising price')
elif prob >= 0.50: print('Moderate demand — price looks reasonable')
else:              print('Low demand — consider lowering price or improving listing')

## 9. Next Steps

| Direction | What to do |
|-----------|-----------|
| **Better target encoding** | Use K-Fold target encoding to prevent leakage (`category_encoders.TargetEncoder`) |
| **Lag features** | Rolling 30/60-day occupancy rate per listing |
| **Supply pressure** | Number of available listings in same neighbourhood + room type on same date |
| **Holiday features** | US federal holidays, local events (`holidays` library) |
| **Lead time** | Days between calendar snapshot date and the booking date |
| **Hyperparameter tuning** | Optuna or `lgb.cv` with Bayesian search |
| **Model explainability** | SHAP via `shap.TreeExplainer(lgb_model)` |
| **Deployment** | Save with `joblib`, serve via FastAPI |

In [ ]:
import joblib
joblib.dump(lgb_model, 'lgb_occupancy_model.pkl')
print('Model saved to lgb_occupancy_model.pkl ✓')

## 10. Bonus: Listing Price Prediction

Parallel regression model predicting nightly price.  
Uses log-transformed price as target to reduce right skew.  
Same time-based train/test split — no leakage.


In [ ]:
print(df_price['price_num'].notna().sum())
print(len(df_price))
# Check which features have NaN in df_price
print("NaN counts in price features:")
print(df_price[PRICE_FEATURES].isnull().sum().sort_values(ascending=False))
print(f"\nTotal rows before dropna: {len(df_price):,}")
print(f"Rows after dropna: {df_price.dropna(subset=PRICE_FEATURES).shape[0]:,}")

In [ ]:
# ── Competitor Pricing Feature via KNN (for Price Model) ──
# For each listing, find the K nearest listings of the same room type
# and compute their average price as a "competitive price" feature.

from sklearn.neighbors import BallTree
import numpy as np

# Step 1: compute each listing's average price across all dates
listing_avg_price = (
    df[df['price_num'].between(20, 800)]
    .groupby('listing_id')['price_num']
    .mean()
    .reset_index()
    .rename(columns={'price_num': 'avg_price'})
)

# Merge avg_price into listings
listings_knn = listings.merge(listing_avg_price, on='listing_id', how='left')
listings_knn = listings_knn.dropna(subset=['latitude','longitude','avg_price','room_type'])
print(f"Listings with valid coords + price: {len(listings_knn):,}")

# Step 2: KNN by room type (only compare same room type = true competitors)
K = 6  # find 6 neighbors, exclude self = 5 competitors
MAX_DIST_KM = 2.0  # only consider listings within 2km
EARTH_RADIUS_KM = 6371.0

competitor_avg = {}

for room_type, group in listings_knn.groupby('room_type'):
    if len(group) < 2:
        # Not enough listings of this type to compare
        for lid in group['listing_id']:
            competitor_avg[lid] = np.nan
        continue

    coords_rad = np.radians(group[['latitude','longitude']].values)
    tree = BallTree(coords_rad, metric='haversine')

    k = min(K, len(group))
    distances, indices = tree.query(coords_rad, k=k)

    for i, (dists, idxs) in enumerate(zip(distances, indices)):
        listing_id = group.iloc[i]['listing_id']

        # Exclude self (index 0) and filter by max distance
        neighbor_mask = (dists[1:] * EARTH_RADIUS_KM) <= MAX_DIST_KM
        valid_neighbors = idxs[1:][neighbor_mask]

        if len(valid_neighbors) == 0:
            competitor_avg[listing_id] = np.nan
        else:
            neighbor_prices = group.iloc[valid_neighbors]['avg_price'].values
            competitor_avg[listing_id] = np.nanmean(neighbor_prices)

# Step 3: map back to listings and then to df_price
listings_knn['competitor_avg_price'] = listings_knn['listing_id'].map(competitor_avg)

mapped = listings_knn['competitor_avg_price'].notna().sum()
print(f"Listings with competitor price: {mapped:,} / {len(listings_knn):,}")
print(f"Average competitor price: ${listings_knn['competitor_avg_price'].mean():.2f}")
print(listings_knn[['listing_id','room_type','avg_price','competitor_avg_price']].head(8).to_string(index=False))

# Step 4: merge into df so price model can use it
df = df.merge(
    listings_knn[['listing_id','competitor_avg_price']],
    on='listing_id', how='left'
)
print(f"\ncompetitor_avg_price added to df")
print(f"Non-null: {df['competitor_avg_price'].notna().sum():,} / {len(df):,}")

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score

df_price = df[df['price_num'].between(20, 800)].copy()
df_price = df_price.sort_values('date')
print(f"Rows with valid price: {len(df_price):,}")

# Removed nbhd_est_income (all NaN) and occupancy-caused features
PRICE_FEATURES = [
    'month', 'day_of_week_n', 'is_weekend', 'quarter',
    'accommodates', 'bedrooms', 'beds', 'minimum_nights',
    'review_scores_rating', 'number_of_reviews', 'reviews_per_month',
    'instant_bookable', 'host_is_superhost',
    'room_type_enc',
    'competitor_avg_price',  #
]
PRICE_FEATURES = [f for f in PRICE_FEATURES if f in df_price.columns]
print(f"Price model features ({len(PRICE_FEATURES)}): {PRICE_FEATURES}")

# Time-based split — no dropna here, imputer handles NaN
price_cutoff = df_price['date'].quantile(0.73)
train_p = df_price[df_price['date'] <= price_cutoff]
test_p  = df_price[df_price['date'] >  price_cutoff]

X_train_p = train_p[PRICE_FEATURES]
y_train_p = np.log1p(train_p['price_num'])
X_test_p  = test_p[PRICE_FEATURES]
y_test_p  = np.log1p(test_p['price_num'])

print(f"Train: {len(X_train_p):,}  |  Test: {len(X_test_p):,}")

# LightGBM handles NaN natively — no imputer needed
lgb_price = lgb.LGBMRegressor(
    n_estimators=500, learning_rate=0.05,
    num_leaves=63, min_child_samples=20,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=1.0,
    random_state=SEED, n_jobs=-1, verbose=-1
)
lgb_price.fit(
    X_train_p, y_train_p,
    eval_set=[(X_test_p, y_test_p)],
    callbacks=[lgb.early_stopping(50, verbose=False)]
)

# Evaluate in original dollar space
pred_price   = np.expm1(lgb_price.predict(X_test_p))
actual_price = np.expm1(y_test_p)

mae = mean_absolute_error(actual_price, pred_price)
r2  = r2_score(actual_price, pred_price)
print(f"\nPrice Model  |  MAE: ${mae:.2f}  |  R2: {r2:.4f}")

# Plots
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(actual_price, pred_price, alpha=0.15, s=4, color='#6366f1')
axes[0].plot([20, 800], [20, 800], 'r--', linewidth=1.2, label='Perfect fit')
axes[0].set_xlabel('Actual Price ($)')
axes[0].set_ylabel('Predicted Price ($)')
axes[0].set_title(f'Price Model: Predicted vs Actual  (MAE=${mae:.0f}, R2={r2:.3f})')
axes[0].set_xlim(20, 800); axes[0].set_ylim(20, 800)
axes[0].legend(); axes[0].grid(alpha=0.3)

fi_p = pd.Series(lgb_price.feature_importances_, index=PRICE_FEATURES).sort_values(ascending=True)
fi_p.plot.barh(ax=axes[1], color='#f59e0b')
axes[1].set_title('Price Model Feature Importance')
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout(); plt.show()

import joblib
joblib.dump(lgb_price, 'lgb_price_model.pkl')
print('Price model saved ✓')

In [ ]:
# ── Verify: which features matter most for Occupancy vs Price ──

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Occupancy model feature importance
fi_occ = pd.Series(
    lgb_model.feature_importances_, index=FEATURES
).sort_values(ascending=True)

fi_occ.plot.barh(ax=axes[0], color='#6366f1')
axes[0].set_title('Occupancy Model — Feature Importance')
axes[0].grid(axis='x', alpha=0.3)

# Price model feature importance
fi_price = pd.Series(
    lgb_price.feature_importances_, index=PRICE_FEATURES
).sort_values(ascending=True)

fi_price.plot.barh(ax=axes[1], color='#f59e0b')
axes[1].set_title('Price Model — Feature Importance')
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

# Print top 5 for each model
print("=== Top 5 features for Occupancy ===")
print(fi_occ.tail(5).sort_values(ascending=False).to_string())

print("\n=== Top 5 features for Price ===")
print(fi_price.tail(5).sort_values(ascending=False).to_string())

# Check missing rate for top features in each model
print("\n=== Missing rate of top Occupancy features ===")
top_occ = fi_occ.tail(5).index.tolist()
print(df[top_occ].isnull().mean().round(3))

print("\n=== Missing rate of top Price features ===")
top_price = fi_price.tail(5).index.tolist()
print(df[top_price].isnull().mean().round(3))

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score

# Filter to sensible price range
df_price = df[df['price_num'].between(20, 800)].copy()
df_price = df_price.sort_values('date')

print(f"Rows with valid price: {len(df_price):,}")

# ── Price features: only static listing attributes ──
# Removed neighbourhood_occ_mean, bookings_last_30d, occ_rate_last_30d
# because occupancy is caused BY price, not the other way around
PRICE_FEATURES = [
    'month', 'day_of_week_n', 'is_weekend', 'quarter',
    'accommodates', 'bedrooms', 'beds', 'minimum_nights',
    'review_scores_rating', 'number_of_reviews', 'reviews_per_month',
    'instant_bookable', 'host_is_superhost',
    'room_type_enc',
    'nbhd_est_income',
]
PRICE_FEATURES = [f for f in PRICE_FEATURES if f in df_price.columns]
print(f"Price model features ({len(PRICE_FEATURES)}): {PRICE_FEATURES}")

# Time-based split
price_cutoff = df_price['date'].quantile(0.73)
train_p = df_price[df_price['date'] <= price_cutoff].dropna(subset=PRICE_FEATURES)
test_p  = df_price[df_price['date'] >  price_cutoff].dropna(subset=PRICE_FEATURES)

X_train_p = train_p[PRICE_FEATURES]
y_train_p = np.log1p(train_p['price_num'])
X_test_p  = test_p[PRICE_FEATURES]
y_test_p  = np.log1p(test_p['price_num'])

print(f"Train: {len(X_train_p):,}  |  Test: {len(X_test_p):,}")

# LightGBM Regressor
lgb_price = lgb.LGBMRegressor(
    n_estimators=500, learning_rate=0.05,
    num_leaves=63, min_child_samples=20,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=1.0,
    random_state=SEED, n_jobs=-1, verbose=-1
)
lgb_price.fit(
    X_train_p, y_train_p,
    eval_set=[(X_test_p, y_test_p)],
    callbacks=[lgb.early_stopping(50, verbose=False)]
)

# Evaluate in original dollar space
pred_price   = np.expm1(lgb_price.predict(X_test_p))
actual_price = np.expm1(y_test_p)

mae = mean_absolute_error(actual_price, pred_price)
r2  = r2_score(actual_price, pred_price)
print(f"\nPrice Model  |  MAE: ${mae:.2f}  |  R2: {r2:.4f}")

# Plots
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(actual_price, pred_price, alpha=0.15, s=4, color='#6366f1')
axes[0].plot([20, 800], [20, 800], 'r--', linewidth=1.2, label='Perfect fit')
axes[0].set_xlabel('Actual Price ($)')
axes[0].set_ylabel('Predicted Price ($)')
axes[0].set_title(f'Price Model: Predicted vs Actual  (MAE=${mae:.0f}, R2={r2:.3f})')
axes[0].set_xlim(20, 800); axes[0].set_ylim(20, 800)
axes[0].legend(); axes[0].grid(alpha=0.3)

fi_p = pd.Series(lgb_price.feature_importances_, index=PRICE_FEATURES).sort_values(ascending=True)
fi_p.plot.barh(ax=axes[1], color='#f59e0b')
axes[1].set_title('Price Model Feature Importance')
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout(); plt.show()

import joblib
joblib.dump(lgb_price, 'lgb_price_model.pkl')
print('Price model saved ✓')

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score

# Filter to sensible price range
df_price = df[df['price_num'].between(20, 800)].copy()
df_price = df_price.sort_values('date')

PRICE_FEATURES = [
    'month', 'day_of_week_n', 'is_weekend', 'quarter',
    'accommodates', 'bedrooms', 'beds', 'minimum_nights',
    'review_scores_rating', 'number_of_reviews', 'reviews_per_month',
    'instant_bookable', 'host_is_superhost',
    'neighbourhood_occ_mean', 'room_type_enc',
    # Demand signals: if a listing was busy last month it can charge more
    'bookings_last_30d', 'occ_rate_last_30d',
    'nbhd_est_income',
]
PRICE_FEATURES = [f for f in PRICE_FEATURES if f in df_price.columns]
print(f"Price model features ({len(PRICE_FEATURES)}): {PRICE_FEATURES}")

# Time-based split (same logic as occupancy model)
price_cutoff = df_price['date'].quantile(0.73)
train_p = df_price[df_price['date'] <= price_cutoff]
test_p  = df_price[df_price['date'] >  price_cutoff]

X_train_p = train_p[PRICE_FEATURES]
y_train_p = np.log1p(train_p['price_num'])
X_test_p  = test_p[PRICE_FEATURES]
y_test_p  = np.log1p(test_p['price_num'])

print(f"Train: {len(X_train_p):,}  |  Test: {len(X_test_p):,}")

# LightGBM Regressor
lgb_price = lgb.LGBMRegressor(
    n_estimators=500, learning_rate=0.05,
    num_leaves=63, min_child_samples=20,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=1.0,
    random_state=SEED, n_jobs=-1, verbose=-1
)
lgb_price.fit(
    X_train_p, y_train_p,
    eval_set=[(X_test_p, y_test_p)],
    callbacks=[lgb.early_stopping(50, verbose=False)]
)

# Evaluate in original dollar space
pred_price   = np.expm1(lgb_price.predict(X_test_p))
actual_price = np.expm1(y_test_p)

mae = mean_absolute_error(actual_price, pred_price)
r2  = r2_score(actual_price, pred_price)
print(f"\nPrice Model  |  MAE: ${mae:.2f}  |  R2: {r2:.4f}")

# Plots
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(actual_price, pred_price, alpha=0.15, s=4, color='#6366f1')
axes[0].plot([20, 800], [20, 800], 'r--', linewidth=1.2, label='Perfect fit')
axes[0].set_xlabel('Actual Price ($)'); axes[0].set_ylabel('Predicted Price ($)')
axes[0].set_title(f'Price Model: Predicted vs Actual  (MAE=${mae:.0f}, R2={r2:.3f})')
axes[0].set_xlim(20, 800); axes[0].set_ylim(20, 800)
axes[0].legend(); axes[0].grid(alpha=0.3)

fi_p = pd.Series(lgb_price.feature_importances_, index=PRICE_FEATURES).sort_values(ascending=True)
fi_p.plot.barh(ax=axes[1], color='#f59e0b')
axes[1].set_title('Price Model Feature Importance')
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout(); plt.show()

import joblib
joblib.dump(lgb_price, 'lgb_price_model.pkl')
print('Price model saved to lgb_price_model.pkl')
